In [1]:
# Record the approved processing rules before changing any data.
import pandas as pd
from pathlib import Path

processing_decisions = pd.DataFrame([
    {
        "table": "observations",
        "issue": "1,085 redundant exact duplicate rows",
        "action": "Remove exact duplicate copies",
        "reason": "Confirmed redundant records",
        "status": "Approved"
    },
    {
        "table": "supplies",
        "issue": "598 redundant exact duplicate rows",
        "action": "Remove exact duplicate copies",
        "reason": "Confirmed redundant records",
        "status": "Approved"
    },
    {
        "table": "All tables with date columns",
        "issue": "Dates stored as strings in raw CSVs",
        "action": "Parse dates explicitly",
        "reason": "Enable reliable date-based analysis",
        "status": "Approved"
    },
    {
        "table": "claims_transactions",
        "issue": "Suspicious 1970 dates and reversed date pairs",
        "action": "Preserve dates and add quality flags",
        "reason": "Avoid unsupported corrections",
        "status": "Approved"
    },
    {
        "table": "medications",
        "issue": "285 reversed date intervals",
        "action": "Preserve dates and add a quality flag",
        "reason": "Avoid guessing the correct dates",
        "status": "Approved"
    },
    {
        "table": "procedures",
        "issue": "2 reversed date intervals",
        "action": "Preserve dates and add a quality flag",
        "reason": "Avoid guessing the correct dates",
        "status": "Approved"
    },
    {
        "table": "All tables",
        "issue": "Missing values",
        "action": "Retain missing values initially",
        "reason": "Missingness requires context-specific treatment",
        "status": "Approved"
    },
    {
        "table": "All tables",
        "issue": "Candidate identifiers",
        "action": "Do not remove records based on repeated IDs alone",
        "reason": "Some identifiers represent groups rather than individual rows",
        "status": "Approved"
    }
])

display(processing_decisions)

,table,issue,action,reason,status
0,observations,"1,085 redundant exact duplicate rows",Remove exact duplicate copies,Confirmed redundant records,Approved
1,supplies,598 redundant exact duplicate rows,Remove exact duplicate copies,Confirmed redundant records,Approved
2,All tables with date columns,Dates stored as strings in raw CSVs,Parse dates explicitly,Enable reliable date-based analysis,Approved
3,claims_transactions,Suspicious 1970 dates and reversed date pairs,Preserve dates and add quality flags,Avoid unsupported corrections,Approved
4,medications,285 reversed date intervals,Preserve dates and add a quality flag,Avoid guessing the correct dates,Approved
5,procedures,2 reversed date intervals,Preserve dates and add a quality flag,Avoid guessing the correct dates,Approved
6,All tables,Missing values,Retain missing values initially,Missingness requires context-specific treatment,Approved
7,All tables,Candidate identifiers,Do not remove records based on repeated IDs alone,Some identifiers represent groups rather than ...,Approved


In [2]:
# Locate the project folders without modifying any raw files.
project_root = Path.cwd()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw" / "synthea"
processed_dir = project_root / "data" / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw data directory:", raw_dir)
print("Processed data directory:", processed_dir)
print("Raw directory exists:", raw_dir.exists())

Project root: C:\Users\USER\Documents\DATA SCIENCE\healthcare-risk-system
Raw data directory: C:\Users\USER\Documents\DATA SCIENCE\healthcare-risk-system\data\raw\synthea
Processed data directory: C:\Users\USER\Documents\DATA SCIENCE\healthcare-risk-system\data\processed
Raw directory exists: True


In [3]:
# Load every CSV table into a dictionary, keeping the source files unchanged.

csv_files = sorted(raw_dir.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in: {raw_dir}"
    )

tables = {
    file_path.stem.lower(): pd.read_csv(file_path, low_memory=False)
    for file_path in csv_files
}

print("Number of tables loaded:", len(tables))

display(
    pd.DataFrame([
        {
            "table": name,
            "rows": len(df),
            "columns": len(df.columns)
        }
        for name, df in tables.items()
    ]).sort_values("table").reset_index(drop=True)
)

Number of tables loaded: 18


,table,rows,columns
0,allergies,5375,15
1,careplans,18976,9
2,claims,629678,31
3,claims_transactions,5547678,33
4,conditions,208363,7
5,devices,33119,7
6,encounters,337227,15
7,imaging_studies,481554,13
8,immunizations,82882,6
9,medications,292451,13


In [4]:
# Create independent working copies for all approved processing operations.
processed_tables = {
    name: df.copy(deep=True)
    for name, df in tables.items()
}

print("Raw tables loaded:", len(tables))
print("Working copies created:", len(processed_tables))

Raw tables loaded: 18
Working copies created: 18


In [5]:
# Remove exact duplicate rows only from the two tables where we confirmed them.
duplicate_removal_summary = []

for table_name in ["observations", "supplies"]:
    df = processed_tables[table_name]

    rows_before = len(df)
    duplicate_copies = int(df.duplicated().sum())

    processed_tables[table_name] = df.drop_duplicates().copy()

    rows_after = len(processed_tables[table_name])

    duplicate_removal_summary.append({
        "table": table_name,
        "rows_before": rows_before,
        "duplicate_copies_removed": duplicate_copies,
        "rows_after": rows_after
    })

duplicate_removal_summary = pd.DataFrame(duplicate_removal_summary)

display(duplicate_removal_summary)

,table,rows_before,duplicate_copies_removed,rows_after
0,observations,4355508,1085,4354423
1,supplies,151616,598,151018


In [6]:
# Define the temporal columns identified during the data quality investigation.
date_columns = {
    "allergies": ["START", "STOP"],
    "careplans": ["START", "STOP"],
    "claims": [
        "CURRENTILLNESSDATE", "SERVICEDATE", "LASTBILLEDDATE1",
        "LASTBILLEDDATE2", "LASTBILLEDDATEP"
    ],
    "claims_transactions": ["FROMDATE", "TODATE"],
    "conditions": ["START", "STOP"],
    "devices": ["START", "STOP"],
    "encounters": ["START", "STOP"],
    "imaging_studies": ["DATE"],
    "immunizations": ["DATE"],
    "medications": ["START", "STOP"],
    "observations": ["DATE"],
    "patients": ["BIRTHDATE", "DEATHDATE"],
    "payer_transitions": ["START_DATE", "END_DATE"],
    "procedures": ["START", "STOP"],
    "supplies": ["DATE"]
}

In [7]:
# Convert known temporal columns to pandas datetime values in the processed copies.
date_conversion_summary = []

for table_name, columns in date_columns.items():
    df = processed_tables[table_name]

    for column in columns:
        if column not in df.columns:
            continue

        original_non_null = df[column].notna()

        parsed_dates = pd.to_datetime(
            df[column],
            errors="coerce",
            utc=True
        )

        date_conversion_summary.append({
            "table": table_name,
            "column": column,
            "original_non_null": int(original_non_null.sum()),
            "parse_failures": int(
                (original_non_null & parsed_dates.isna()).sum()
            ),
            "missing_after_conversion": int(parsed_dates.isna().sum())
        })

        df[column] = parsed_dates

date_conversion_summary = pd.DataFrame(date_conversion_summary)

display(
    date_conversion_summary[
        date_conversion_summary["parse_failures"] > 0
    ]
)

print("Date conversion complete.")
print(
    "Columns with parse failures:",
    int((date_conversion_summary["parse_failures"] > 0).sum())
)

,table,column,original_non_null,parse_failures,missing_after_conversion


Date conversion complete.
Columns with parse failures: 0


In [8]:
# Flag suspicious transaction dates while preserving the original date values.
transactions = processed_tables["claims_transactions"]

transactions["FLAG_EPOCH_TODATE"] = (
    transactions["TODATE"]
    == pd.Timestamp("1970-01-01", tz="UTC")
)

transactions["FLAG_REVERSED_DATE_ORDER"] = (
    transactions["FROMDATE"].notna()
    & transactions["TODATE"].notna()
    & (transactions["FROMDATE"] > transactions["TODATE"])
)

# Flag medication records whose stop date precedes their start date.
medications = processed_tables["medications"]

medications["FLAG_REVERSED_DATE_ORDER"] = (
    medications["START"].notna()
    & medications["STOP"].notna()
    & (medications["STOP"] < medications["START"])
)

# Apply the same check to procedures.
procedures = processed_tables["procedures"]

procedures["FLAG_REVERSED_DATE_ORDER"] = (
    procedures["START"].notna()
    & procedures["STOP"].notna()
    & (procedures["STOP"] < procedures["START"])
)

print("Transaction rows with epoch TODATE:",
      int(transactions["FLAG_EPOCH_TODATE"].sum()))

print("Transaction rows with reversed date order:",
      int(transactions["FLAG_REVERSED_DATE_ORDER"].sum()))

print("Medication rows with reversed date order:",
      int(medications["FLAG_REVERSED_DATE_ORDER"].sum()))

print("Procedure rows with reversed date order:",
      int(procedures["FLAG_REVERSED_DATE_ORDER"].sum()))

Transaction rows with epoch TODATE: 372215
Transaction rows with reversed date order: 371531
Medication rows with reversed date order: 285
Procedure rows with reversed date order: 2


In [9]:
# Save each processed table to its own CSV file.
for table_name, df in processed_tables.items():
    output_path = processed_dir / f"{table_name}.csv"
    df.to_csv(output_path, index=False)

print(f"Saved {len(processed_tables)} processed tables to:")
print(processed_dir)

Saved 18 processed tables to:
C:\Users\USER\Documents\DATA SCIENCE\healthcare-risk-system\data\processed


In [10]:
# Confirm that the processed CSV files exist and compare their row counts.
verification_rows = []

for table_name, df in processed_tables.items():
    output_path = processed_dir / f"{table_name}.csv"

    verification_rows.append({
        "table": table_name,
        "processed_rows": len(df),
        "file_exists": output_path.exists(),
        "file_size_kb": (
            round(output_path.stat().st_size / 1024, 2)
            if output_path.exists() else None
        )
    })

verification_df = pd.DataFrame(verification_rows)

display(verification_df)

assert verification_df["file_exists"].all(), (
    "At least one processed CSV file is missing."
)

print("Verification passed: all processed table files exist.")

,table,processed_rows,file_exists,file_size_kb
0,allergies,5375,True,1036.03
1,careplans,18976,True,4174.13
2,claims,629678,True,255771.17
3,claims_transactions,5547678,True,2580618.45
4,conditions,208363,True,37520.08
5,devices,33119,True,7601.51
6,encounters,337227,True,113791.44
7,imaging_studies,481554,True,167910.12
8,immunizations,82882,True,11731.98
9,medications,292451,True,80084.46


Verification passed: all processed table files exist.
